# 08 — From VLMs to learning on an SO-101 arm

A VLM is a useful semantic sensor, not a low-level robot controller. A typical VLA policy consumes camera frame(s), language instruction, and proprioception (joint positions/velocities, gripper state), then predicts a joint-space action or a chunk of actions. On an SO-101, action units, motor calibration, observation timing, camera geometry, and control frequency must be explicit.

## Recommended development ladder

1. **Instrument and calibrate:** verify joint limits, sign, zero offsets, gripper range, camera placement, timestamp synchronization, and emergency stop.
2. **Collect demonstrations:** teleoperate a leader/follower; save observations, states, actions, task text, episode boundaries, and metadata. Repeat tasks across object poses, lighting, and resets.
3. **Offline data checks:** plot state/action ranges, detect dropped frames and discontinuities, group train/test by episode, ensure no unsafe demonstration labels.
4. **Behavior clone first:** train a supervised policy to predict next action/action chunk; evaluate held-out episodes and recovery cases. This is usually safer than beginning with online RL.
5. **Use a pretrained VLA or modular baseline:** compare a small policy, SmolVLA/OpenVLA-compatible model if its action interface fits, and a conventional low-level controller. An image-text model alone is not a VLA.
6. **Simulation/offline policy evaluation:** replay, perturb initial conditions, enforce action bounds, test latency and failures.
7. **Constrained real-world validation:** workspace barriers, low speed/torque, soft objects, one operator at the emergency stop, short action horizon, human confirmation, and a rollback policy.
8. **Only then consider RL:** start with conservative offline RL or a carefully bounded simulation policy. Online exploration on a physical arm can damage equipment or injure people.

LeRobot's SO-101 documentation describes a 6-motor follower arm, motor setup/calibration, and recording workflows. Current docs and wiring details should be checked for your exact hardware revision.

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd

# Example trajectory table shape; values are placeholders, not SO-101 calibration data.
joint_names = ['shoulder_pan', 'shoulder_lift', 'elbow_flex', 'wrist_flex', 'wrist_roll', 'gripper']
q_min = torch.tensor([-1.0, -1.0, -1.0, -1.0, -1.0, 0.0])
q_max = torch.tensor([1.0, 1.0, 1.0, 1.0, 1.0, 1.0])
raw_action = torch.tensor([0.2, -0.4, 0.1, 0.3, 0.0, 0.8])
bounded_action = torch.maximum(torch.minimum(raw_action, q_max), q_min)
assert torch.all(bounded_action >= q_min) and torch.all(bounded_action <= q_max)
pd.DataFrame({'joint': joint_names, 'command_example': bounded_action.numpy(), 'warning': 'placeholder bounds only'})

## Behavior cloning objective and a simple RL return

For continuous actions, supervised behavior cloning commonly minimizes MSE or a probabilistic action loss. Action chunking predicts a short future sequence, often reducing inference calls but making stale plans and latency important. RL introduces rewards, transitions, and exploration; reward hacking and sparse success signals are common pitfalls. The code below only illustrates tensor arithmetic and must not control hardware.

In [ ]:
torch.manual_seed(4)
predicted = torch.randn(8, 6, requires_grad=True)  # batch x 6 joint deltas
demonstrated = torch.randn(8, 6)
bc_loss = F.mse_loss(predicted, demonstrated)
bc_loss.backward()
print('behavior-cloning loss:', float(bc_loss), 'gradient exists:', predicted.grad is not None)

rewards = torch.tensor([0.0, 0.0, 0.0, 1.0])
gamma = 0.99
discounted_return = sum((gamma ** t) * rewards[t] for t in range(len(rewards)))
print('illustrative discounted return:', float(discounted_return))

## RL design choices and safety gates

- Define state `s_t` (camera(s), joint state, task), action `a_t` (absolute target vs delta; radians vs normalized; gripper convention), transition, reward, termination, and reset protocol before training.
- Offline RL avoids risky online exploration but can fail outside the logged action distribution. Behavior cloning is a strong baseline and data-quality diagnostic.
- For initial RL, prefer a simulation with randomized camera/object pose and realistic action limits. Make success measurable and penalize unsafe contact, excessive speed, and workspace violation.
- A VLM-generated plan can be wrong or adversarially influenced by image text. Treat it as untrusted input; validate all targets/actions in a deterministic controller.
- Add a watchdog for stale observations, deadline misses, NaNs, action clipping, and communications faults. On any failure, stop motion safely.
- Report task success with confidence intervals across independent resets; measure collisions, drops, intervention rate, and time-to-completion, not just reward.

## Data record for an episode

At minimum: episode ID, timestamp, camera frame(s), calibrated joint positions, commanded and measured actions, task text, success/failure label, reset metadata, and software/model version. Respect privacy, retain only approved data, and ensure the train/test split is at episode level.

Sources: SO-101 setup https://huggingface.co/docs/lerobot/so101 ; LeRobot https://github.com/huggingface/lerobot ; OpenVLA https://openvla.github.io/ ; SmolVLA https://arxiv.org/abs/2506.01844